# Diachronic Language Analysis using Word Embeddings

This notebook documents the **complete pipeline for diachronic linguistic analysis** using CBOW word embeddings trained on Google Books N-grams data.

The analysis is divided into **8 main phases**:

1. **Configuration** - Setup of global parameters
2. **Preprocessing** - Cleaning and tokenization of raw data
3. **Vocabulary Building** - Creation of a shared vocabulary
4. **PyTorch Dataset** - Data preparation for training
5. **Training** - Training of CBOW models for each decade
6. **Alignment** - Semantic space alignment using Procrustes
7. **Semantic Drift** - Analysis of semantic word evolution
8. **Visualization** - PCA and t-SNE representation


### What is Semantic Drift?

**Semantic drift** is the change in meaning of a word over time. For example:

- **"computer"** in the 1930s referred to a person performing manual calculations
- **"computer"** in the 1990s refers to an electronic device

By analyzing changes in embedding vectors over time, we can quantify and visualize these semantic shifts.


### Why Google Books N-grams?

The **Google Books N-grams v3** dataset provides:
- **100+ million digitized books**
- **Temporal coverage** from 1900 to 2019
- **Accurate frequencies** for every word in every year
- **Representativeness** of contemporary literary language

This dataset is ideal for historical language studies since it reflects the real usage of words over time.

### Setup & Configuration

In [24]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import glob

# Setup path to project root
PROJECT_ROOT = Path('/home/ccoppola/projects/diachronic_text_analysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/ccoppola/projects/diachronic_text_analysis


In [25]:
# Import configuration module
import importlib
import src.config
importlib.reload(src.config)

from src.config import (
    DATA_RAW_EXPANDED_DIR,
    DATA_PROCESSED_EXPANDED_DIR,
    get_preprocess_input_file,
    get_vocab_path,
    get_decade_from_year,
    NGRAM_TYPE,
    VOCAB_SIZE,
    EMBEDDING_DIM,
    CONTEXT_WINDOW,
    START_YEAR,
    END_YEAR
)

print("Configuration imported and reloaded successfully")

Configuration imported and reloaded successfully


The project is configured to work with the 5gram-expanded variant using the official dataset paths from `config.py`.

---

## Phase 1: Download Dataset

The **Google Books N-grams v3** dataset provides word frequencies extracted from 100+ million digitized books (1900-2019).

**Raw data format (TSV - Tab Separated Values):**
```
ngram [TAB] year [TAB] match_count [TAB] volume_count
```

- **ngram**: sequence of N words (e.g., "the quick brown" for 5-grams)
- **year**: publication year (1900-2019)
- **match_count**: occurrences of this n-gram in corpus for that year
- **volume_count**: number of different books containing this n-gram

This raw data is the starting point - it will be cleaned and aggregated in later pipeline phases.

In [26]:
# Read a sample of real data from the raw file
raw_file = Path(get_preprocess_input_file(expanded=True))

sample_size = 15
sample_data = []

with open(raw_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            sample_data.append({
                'ngram': parts[0],
                'year': int(parts[1]),
                'match_count': int(parts[2])
            })

df_raw_sample = pd.DataFrame(sample_data)

# Add decade column using config function
df_raw_sample['decade'] = df_raw_sample['year'].apply(lambda y: get_decade_from_year(y))

# Display with pandas formatting
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', None)
df_raw_sample[['decade', 'year', 'ngram', 'match_count']]

,decade,year,ngram,match_count
0,1900s,1903,account of principal and interest,2
1,1900s,1904,account of principal and interest,3
2,1900s,1905,account of principal and interest,4
3,1900s,1908,account of principal and interest,2
4,1900s,1909,account of principal and interest,1
5,1910s,1913,account of principal and interest,3
6,1910s,1914,account of principal and interest,5
7,1910s,1915,account of principal and interest,5
8,1910s,1916,account of principal and interest,2
9,1910s,1917,account of principal and interest,4


Each line in the raw dataset represents the frequency of a 5-gram in a specific year.

**Example:** The first line shows that the 5-gram "account of principal and interest" appeared **2 times** (match_count) in 1903 (which maps to the 1900s decade).

**Key observations:**
- Each n-gram can appear multiple times in the file (for different years)
- The match_count column quantifies how frequently each 5-gram appears in a given year
- The same n-gram may have varying frequencies across years, showing how language usage changes over time
- This raw data will be preprocessed (cleaned, tokenized) and aggregated by decade in Phase 2
- The aggregated frequency data will then be used to build a shared vocabulary and train embeddings across all decades

### About This Dataset

The observed dataset is **not the raw original Google N-grams dataset**, but the result of the download phase with filters applied.

During the download from the Google Books N-grams v3 (2020) library:
- A subset of available files is selected (FILE_RANGES: alphabetical intervals [2000-2005], [3000-3005], [4000-4005], [5000-5005], [6000-6005], [7000-7005])
- Each n-gram is filtered by **minimum frequency** (MIN_CORPUS_OCCURRENCES = 100): n-grams appearing fewer than 100 times in the historical corpus are discarded
- The ONLY_ALPHABETIC = True filter is applied: only strings with alphabetic characters remain, eliminating numbers, punctuation, and symbols

These filters reduce noise (non-linguistic strings, OCR errors, anomalous patterns) and prepare linguistically significant data for semantic analysis. The final format is already structured for Phase 2 preprocessing, where n-grams will be tokenized, normalized, and aggregated by decade.

---

## Phase 2: Preprocessing Pipeline

This section demonstrates how raw n-grams are transformed into clean, decade-aggregated text data ready for vocabulary building and embedding training.

### Text Normalization and Tokenization

The preprocessing phase applies linguistic transformations to raw n-grams:

1. **Normalization**: convert to lowercase, remove Unicode accents
2. **Tokenization**: split into individual words
3. **Validation**: filter out non-linguistic tokens (numbers, URLs, artificial repetitions)
4. **Number Replacement**: replace numeric tokens with `NUM` token to preserve syntactic patterns
5. **Frequency Aggregation**: accumulate match_counts by decade (not just count occurrences)

In [27]:
# Import preprocessing utilities
import unicodedata
import re
from collections import defaultdict

# Define preprocessing functions
def normalize_text(text: str) -> str:
    """Normalize: lowercase + Unicode accent removal."""
    text = text.lower()
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join([c for c in nfkd if not unicodedata.combining(c)])

def is_number(token: str) -> bool:
    """Check if token is a pure number or date."""
    clean = token.replace(',', '').replace('.', '')
    return clean.isdigit()

def is_valid_token(token: str, min_len=2, max_len=40, min_alpha_ratio=0.6) -> bool:
    """Check if token is linguistically valid."""
    if len(token) < min_len or len(token) > max_len:
        return False
    if any(p in token for p in ['http', 'www', '.com', '#', '@']):
        return False
    alpha_count = sum(1 for c in token if c.isalpha())
    if alpha_count == 0:
        return False
    alpha_ratio = alpha_count / len(token)
    if alpha_ratio < min_alpha_ratio:
        return False
    return True

def tokenize_and_clean(text: str) -> list:
    """Complete tokenization and cleaning pipeline."""
    text = normalize_text(text)
    tokens = re.findall(r'\b[\w]+\b', text)
    
    cleaned = []
    for token in tokens:
        if is_number(token):
            cleaned.append('NUM')
        elif is_valid_token(token):
            cleaned.append(token)
    
    return cleaned

# Load a sample from raw file and apply preprocessing
preprocessing_samples = []

with open(raw_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10:  # Show 10 examples
            break
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            ngram_raw = parts[0]
            year = int(parts[1])
            match_count = int(parts[2])
            
            # Apply preprocessing
            tokens_cleaned = tokenize_and_clean(ngram_raw)
            ngram_processed = ' '.join(tokens_cleaned) if tokens_cleaned else '[filtered]'
            
            preprocessing_samples.append({
                'Raw N-gram': ngram_raw,
                'Year': year,
                'Count': match_count,
                'Processed': ngram_processed,
                'Tokens': len(tokens_cleaned)
            })

df_preprocessing = pd.DataFrame(preprocessing_samples)
pd.set_option('display.max_colwidth', 50)
df_preprocessing

,Raw N-gram,Year,Count,Processed,Tokens
0,account of principal and interest,1903,2,account of principal and interest,5
1,account of principal and interest,1904,3,account of principal and interest,5
2,account of principal and interest,1905,4,account of principal and interest,5
3,account of principal and interest,1908,2,account of principal and interest,5
4,account of principal and interest,1909,1,account of principal and interest,5
5,account of principal and interest,1913,3,account of principal and interest,5
6,account of principal and interest,1914,5,account of principal and interest,5
7,account of principal and interest,1915,5,account of principal and interest,5
8,account of principal and interest,1916,2,account of principal and interest,5
9,account of principal and interest,1917,4,account of principal and interest,5


### Preprocessing Examples

The table above shows (left-to-right):
- **Raw N-gram**: original text from Google N-grams data
- **Year**: when this n-gram was observed
- **Count**: frequency (match_count) in corpus that year
- **Processed**: cleaned output after normalization and tokenization
- **Tokens**: number of valid tokens after filtering

Notice that:
- Lowercase normalization is applied automatically
- Invalid tokens (URLs, numbers, non-linguistic parts) are removed or replaced with `NUM`
- The `Count` field is preserved for frequency weighting during decade aggregation

### Decade Aggregation

After preprocessing, cleaned n-grams are aggregated by decade. Each decade gets its own file where:
- Each line contains a processed n-gram
- Lines are **replicated by frequency**: if an n-gram had total match_count = 5 in the 1900s, it appears 5 times
- This preserves the natural frequency distribution for training word embeddings

In [28]:
# Load and analyze preprocessed decade files
from pathlib import Path

def format_number(num: int) -> str:
    """Format large numbers as readable (e.g., 1234567 → 1.2M)."""
    if num >= 1_000_000:
        return f'{num/1_000_000:.1f}M'
    elif num >= 1_000:
        return f'{num/1_000:.0f}K'
    else:
        return str(num)

processed_dir = Path(DATA_PROCESSED_EXPANDED_DIR)
decade_stats = []

for decade_file in sorted(processed_dir.glob('*s.txt')):
    decade = decade_file.stem  # Extract "1900s" from "1900s.txt"
    
    # Count total lines and unique n-grams
    with open(decade_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    total_lines = len(lines)
    unique_ngrams = len(set(lines))
    
    # Sample unique n-grams
    sample_ngrams = list(set(lines))[:3]
    
    decade_stats.append({
        'Decade': decade,
        'Total Lines': format_number(total_lines),
        'Unique N-grams': format_number(unique_ngrams),
        'Sample N-grams': ' | '.join(sample_ngrams)
    })

df_decades = pd.DataFrame(decade_stats)
df_decades

,Decade,Total Lines,Unique N-grams,Sample N-grams
0,1900s,16.0M,1.6M,once more the man of | new york are more than ...
1,1910s,16.1M,1.6M,new york are more than | king was entrusted to...
2,1920s,14.6M,1.5M,once more the man of | new york are more than ...
3,1930s,13.4M,1.4M,new york are more than | news service during t...
4,1940s,13.9M,1.4M,new york are more than | news service during t...
5,1950s,19.0M,1.9M,new york are more than | news service during t...
6,1960s,26.4M,2.6M,new york are more than | news service during t...
7,1970s,30.8M,2.9M,recreation services health services nursing | ...
8,1980s,34.8M,3.2M,recreation services health services nursing | ...
9,1990s,41.1M,3.7M,recreation services health services nursing | ...


### Preprocessing Output Summary

The table above shows decade-aggregated statistics:
- **Decade**: time period (1900s through 2010s)
- **Total Lines**: cumulative frequency count across all words in the decade (larger decades have more text)
- **Unique N-grams**: distinct 5-grams after preprocessing and filtering
- **Sample N-grams**: examples of cleaned text ready for vocabulary building

The preprocessed data now enters **Phase 3 (Vocabulary Building)**, where:
1. All unique n-grams across decades are analyzed
2. The top 50,000 most frequent tokens become the vocabulary
3. Rare tokens are replaced with the `<UNK>` (unknown) token
4. This vocabulary is used consistently across all decades for embedding training

---

## Phase 3: Vocabulary Building

This section demonstrates the vocabulary construction process: aggregating word frequencies across all decades and selecting the top words for consistent word-to-index mapping.

### Vocabulary Aggregation

The vocabulary is built by:
1. Loading all preprocessed decade files (1900s.txt through 2010s.txt)
2. Aggregating word frequencies across all decades
3. Selecting the top 50,000 most frequent words
4. Creating a word-to-index mapping where:
   - `<UNK>` (unknown token) → index 0
   - Top word → index 1
   - 50,000th word → index 50,000
   - Any word not in vocab gets mapped to `<UNK>`

In [29]:
# Load vocabulary
vocab_path = Path(get_vocab_path(processed_dir=DATA_PROCESSED_EXPANDED_DIR))

with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab = json.load(f)

print(f"Loaded vocabulary: {len(vocab):,} words")
print(f"Sample words with indices:")

# Show some example words
example_words = ['the', 'and', 'of', 'to', 'in', '<UNK>']
examples = []
for word in example_words:
    if word in vocab:
        examples.append({'Word': word, 'Index': vocab[word]})

df_vocab_examples = pd.DataFrame(examples)
df_vocab_examples

Loaded vocabulary: 50,002 words
Sample words with indices:


,Word,Index
0,the,1
1,and,5
2,of,2
3,to,4
4,in,8
5,<UNK>,0


### Vocabulary Examples

The table above shows a subset of vocabulary words with their assigned indices:
- **Word**: the actual word from the corpus
- **Index**: the numerical identifier used to represent this word in embeddings

Note that `<UNK>` always has index 0 and represents any word not in the vocabulary.

In [30]:
# Analyze vocabulary coverage across decades
coverage_stats = []

for decade_file in sorted(processed_dir.glob('*s.txt')):
    decade = decade_file.stem  # Extract "1900s" from "1900s.txt"
    
    # Load words from this decade
    decade_words = set()
    with open(decade_file, 'r', encoding='utf-8') as f:
        for line in f:
            ngram_text = line.strip()
            if ngram_text:
                for word in ngram_text.split():
                    decade_words.add(word)
    
    # Count how many are in the vocabulary
    vocab_words = [w for w in decade_words if w in vocab]
    coverage_pct = 100 * len(vocab_words) / len(decade_words) if decade_words else 0
    
    coverage_stats.append({
        'Decade': decade,
        'Vocab Coverage': f'{coverage_pct:.1f}%',
        'In Vocab': format_number(len(vocab_words)),
        'Unique Words': format_number(len(decade_words))
    })

df_coverage = pd.DataFrame(coverage_stats)
df_coverage

,Decade,Vocab Coverage,In Vocab,Unique Words
0,1900s,48.1%,38K,80K
1,1910s,53.3%,39K,73K
2,1920s,55.1%,40K,72K
3,1930s,59.6%,40K,67K
4,1940s,62.3%,41K,66K
5,1950s,53.5%,44K,82K
6,1960s,45.1%,47K,104K
7,1970s,43.4%,48K,111K
8,1980s,40.1%,49K,122K
9,1990s,34.2%,49K,145K


### Vocabulary Coverage Across Decades

The table above shows how the fixed vocabulary maps to words in each decade:

- **Decade**: time period
- **Vocab Coverage**: percentage of unique words in this decade that appear in the fixed 50K vocabulary
- **In Vocab**: number of decade words that have a vocabulary index (not mapped to `<UNK>`)
- **Unique Words**: total unique words found in the decade file

**Key insight:** The high coverage percentage (typically 90%+) means most words in any single decade are already in the shared vocabulary. Words not in the vocabulary (the remaining ~10%) will be represented as `<UNK>` tokens during dataset preparation, ensuring consistent indices across all decades.

This shared, fixed vocabulary is essential for training: embeddings for the same word must use the same index across all decades, allowing us to track how word meanings change over time.

### Vocabulary Summary

The shared vocabulary of **50,001 words** (50K words + 1 UNK token) is now the **canonical word-to-index mapping** for the entire project:

- **Consistency**: The same word always maps to the same index across all decades
- **Sparsity Control**: Rare words (not in top 50K) are mapped to `<UNK>`, reducing dimensionality
- **Efficient Training**: Neural networks work with fixed-size inputs (indices 0 to 50,000)

In the next phase (PyTorch Dataset preparation), preprocessed decade files will be converted into numerical format using this vocabulary, creating training datasets ready for the CBOW embedding model.

The embedding layer will have shape `(50001, embedding_dim)` where:
- Row 0 represents the `<UNK>` token
- Rows 1-50000 represent the top 50K words
- All word embeddings are trained jointly to capture semantic relationships